In [1]:
stop

NameError: name 'stop' is not defined

In [17]:
import os

_new_directory = "/net/scratch/pkwon2/more_tips_test_set" ## change to correct directory

print(f"The current working directory is: {os.getcwd()}")

os.chdir(_new_directory)
print(f"Changed working directory to: {os.getcwd()}")

def create_directory_structure(base_directory):
    try:
        os.makedirs(os.path.join(base_directory, 'input'), exist_ok=True)
        os.makedirs(os.path.join(base_directory, 'outputs'), exist_ok=True)
        print(f"Directory structure '{base_directory}/input' and '{base_directory}/output' created successfully.")
    except Exception as e:
        print(f"Error creating directory structure '{base_directory}': {e}")

if __name__ == "__main__":
    base_directory = "diffusion"
    create_directory_structure(base_directory)

The current working directory is: /net/scratch/pkwon2/more_tips_test_set
Changed working directory to: /net/scratch/pkwon2/more_tips_test_set
Directory structure 'diffusion/input' and 'diffusion/output' created successfully.


In [ ]:
# #!/usr/bin/env python3
# """
# From each input PDB, produce:
#   • <basename>_ca_com.pdb  – single ORI at CA centre-of-mass
# Existing ORI pseudo-atoms are removed so duplicates never appear.
# """
# import os
# import glob

# # ── helpers ────────────────────────────────────────────────────────────────────
# def compute_com(coords):
#     n = len(coords)
#     return (sum(c[0] for c in coords) / n,
#             sum(c[1] for c in coords) / n,
#             sum(c[2] for c in coords) / n)

# def build_hetatm_line(x, y, z, serial=9999):
#     return (f"HETATM{serial:5d}"
#             f" {'ORI':>4s}"
#             f" {'ORI':>3s}"
#             f" {'X'}"
#             f"{13:4d}"             # residue number
#             f"    {x:8.3f}{y:8.3f}{z:8.3f}"
#             f"{1.00:6.2f}{1.00:6.2f}"
#             f"{'X':>12s}")

# # ── core ───────────────────────────────────────────────────────────────────────
# def write_oriented_pdb(src, atom):
#     coords, clean_lines = [], []
#     with open(src) as fh:
#         for ln in fh:
#             # drop any previous ORI lines so we don’t duplicate them
#             if ln[12:16].strip() == "ORI":
#                 continue
#             clean_lines.append(ln)

#             # collect CA coords
#             if ln.startswith(("ATOM", "HETATM")) and ln[12:16].strip() == atom:
#                 try:
#                     coords.append((float(ln[30:38]),
#                                    float(ln[38:46]),
#                                    float(ln[46:54])))
#                 except ValueError:
#                     pass

#     if not coords:
#         print(f"[SKIP] {os.path.basename(src)} – no {atom} atoms")
#         return

#     com_x, com_y, com_z = compute_com(coords)
#     hetatm = build_hetatm_line(com_x, com_y, com_z)

#     # tidy ending
#     if clean_lines and not clean_lines[-1].endswith("\n"):
#         clean_lines[-1] += "\n"
#     if clean_lines and not clean_lines[-1].strip().startswith("END"):
#         clean_lines.append("END\n")

#     out_path = src.replace(".pdb", f"_{atom.lower()}_com.pdb")
#     with open(out_path, "w") as out:
#         out.writelines(clean_lines)
#         out.write(hetatm + "\n")

#     print(f"[WRITE] {os.path.basename(out_path)} – {atom} COM = "
#           f"({com_x:.3f}, {com_y:.3f}, {com_z:.3f})")

# def process_pdb(pdb_path):
#     write_oriented_pdb(pdb_path, "CA")

# # ── entry point ────────────────────────────────────────────────────────────────
# if __name__ == "__main__":
#     pdb_dir = "/net/scratch/pkwon2/more_tips_test_set"      # adjust if needed
#     for pdb in glob.glob(os.path.join(pdb_dir, "*.pdb")):
#         process_pdb(pdb)

In [ ]:
#!/usr/bin/env python3
"""
From each input PDB, produce:
  • <basename>_ca_plane_ori.pdb  – single ORI pseudo-atom offset along the minor PCA axis
Existing ORI pseudo-atoms are removed so duplicates never appear.

Minor axis is determined as the eigenvector of the calcium HETATM covariance matrix with the smallest eigenvalue; we then offset 10 Å along this vector toward glutamate oxygens (if present).
"""
import os
import glob
import numpy as np

# ── helpers ────────────────────────────────────────────────────────────────────
def compute_com(coords):
    """Compute center-of-mass of list of (x, y, z)."""
    pts = np.array(coords)
    return np.mean(pts, axis=0)


def compute_minor_axis(coords):
    """Perform PCA on coords and return eigenvector with smallest eigenvalue (unit)."""
    pts = np.array(coords)
    centroid = compute_com(pts)
    centered = pts - centroid
    cov = centered.T @ centered
    # eigen decomposition
    vals, vecs = np.linalg.eigh(cov)
    # minor axis: eigenvector with smallest eigenvalue
    minor = vecs[:, np.argmin(vals)]
    return minor / np.linalg.norm(minor)


def build_hetatm_line(x, y, z, serial=9999):
    return (f"HETATM{serial:5d}"
            f" {'ORI':>4s}"
            f" {'ORI':>3s}"
            f" {'X'}"
            f"{13:4d}"
            f"    {x:8.3f}{y:8.3f}{z:8.3f}"
            f"{1.00:6.2f}{1.00:6.2f}"
            f"{'X':>12s}")

# ── core ───────────────────────────────────────────────────────────────────────
def write_oriented_pdb(src, axis_offset=12.0):
    ca_coords, glu_coords, clean_lines = [], [], []

    with open(src) as fh:
        for ln in fh:
            if ln[12:16].strip() == 'ORI':
                continue
            clean_lines.append(ln)

            # calcium HETATM by element
            if ln.startswith('HETATM') and ln[76:78].strip() == 'Ca':
                try:
                    ca_coords.append((float(ln[30:38]), float(ln[38:46]), float(ln[46:54])))
                except ValueError:
                    pass
            # glutamate side-chain oxygens
            if ln.startswith(('ATOM','HETATM')) and ln[17:20].strip() == 'GLU' and ln[12:16].strip() in ('OE1','OE2'):
                try:
                    glu_coords.append((float(ln[30:38]), float(ln[38:46]), float(ln[46:54])))
                except ValueError:
                    pass

    if not ca_coords:
        print(f"[SKIP] {os.path.basename(src)} – no calcium HETATM found")
        return

    centroid = compute_com(ca_coords)
    minor_axis = compute_minor_axis(ca_coords)

    # orient minor axis toward glutamates if present
    if glu_coords:
        glu_com = compute_com(glu_coords)
        if np.dot(glu_com - centroid, minor_axis) < 0:
            minor_axis = -minor_axis

    # offset position along minor axis
    ori_pos = centroid + minor_axis * axis_offset
    hetatm = build_hetatm_line(*ori_pos)

    # ensure proper ending
    if clean_lines and not clean_lines[-1].endswith("\n"):
        clean_lines[-1] += "\n"
    if clean_lines and not clean_lines[-1].strip().startswith('END'):
        clean_lines.append('END\n')

    out_path = src.replace('.pdb', '_ca_plane_ori.pdb')
    with open(out_path, 'w') as out:
        out.writelines(clean_lines)
        out.write(hetatm + '\n')

    cx, cy, cz = centroid
    print(f"[WRITE] {os.path.basename(out_path)} – centroid ({cx:.3f}, {cy:.3f}, {cz:.3f}); ORI at ({ori_pos[0]:.3f}, {ori_pos[1]:.3f}, {ori_pos[2]:.3f})")


def process_pdb(pdb_path):
    write_oriented_pdb(pdb_path)

if __name__ == '__main__':
    pdb_dir = '/net/scratch/pkwon2/more_tips_test_set'
    for pdb in glob.glob(os.path.join(pdb_dir, '*.pdb')):
        process_pdb(pdb)

In [ ]:
#!/usr/bin/env python3
import os
import glob

# Atom names to keep
KEEP_ATOMS = {"OE1", "OE2", "CD", "CG"}

def process_pdb(input_path, output_path):
    """
    Read input_path PDB, filter ATOM records to only KEEP_ATOMS,
    renumber them A1, A2,..., drop MODEL/ENDMDL/END records,
    and write to output_path.
    """
    count = 1
    with open(input_path) as infile, open(output_path, 'w') as outfile:
        for line in infile:
            # Determine record name in cols 1-6
            rec_name = line[:6].strip()
            # Skip MODEL, ENDMDL, and END records
            if rec_name in ("MODEL", "ENDMDL", "END"):
                continue
            # Process ATOM lines
            if line.startswith("ATOM  "):
                atom_name = line[12:16].strip()
                if atom_name not in KEEP_ATOMS:
                    continue
                # Assign new ID 'A<count>' right-aligned in cols 7-11
                new_id = f"A{count}"
                count += 1
                id_field = new_id.rjust(5)
                outfile.write(line[:6] + id_field + line[11:])
            else:
                # Keep HETATM and other lines unchanged
                outfile.write(line)

# In Jupyter notebook: process all *_ca_plane_ori.pdb in current directory
cwd = os.getcwd()
pattern = os.path.join(cwd, "*_ca_plane_ori.pdb")
pdb_paths = glob.glob(pattern)

if not pdb_paths:
    print(f"No files matching '*_ca_plane_ori.pdb' in {cwd}")
else:
    for pdb_file in pdb_paths:
        base = os.path.basename(pdb_file)
        out_name = base.replace(".pdb", "_linesremoved.pdb")
        out_path = os.path.join(cwd, out_name)
        print(f"Processing {base} -> {out_name}")
        process_pdb(pdb_file, out_path)
    print("All matching files processed.")

In [ ]:
#!/usr/bin/env python3
import os
import glob
import string

# Directory containing your PDB files
directory = "/net/scratch/pkwon2/more_tips_test_set"

# Define the atom-ordering within each 4-line chunk
atom_order = ["OE1", "OE2", "CD", "CG"]

for infile in glob.glob(os.path.join(directory, "*linesremoved.pdb")):
    base = os.path.basename(infile)
    outfile = os.path.join(
        directory,
        base.replace(".pdb", "_chunked.pdb")
    )
    print(f"→ {base}  →  {os.path.basename(outfile)}")

    # Read all lines
    with open(infile, "r") as f:
        lines = f.readlines()

    # Split into successive groups of 4 lines
    chunks = [lines[i : i + 4] for i in range(0, len(lines), 4)]

    new_lines = []
    for idx, chunk in enumerate(chunks, start=1):
        # If any line in this chunk is a HETATM record, leave the entire chunk untouched
        if any(line.startswith("HETATM") for line in chunk):
            new_lines.extend(chunk)
            continue

        # Letter for chain ID: A, B, C, ...
        chain_id = string.ascii_uppercase[idx - 1]
        # Numeric residue sequence
        res_seq = idx

        # Attempt to sort by atom name (chars 14–16 → line[13:16])
        try:
            chunk_sorted = sorted(
                chunk,
                key=lambda L: atom_order.index(L[13:16].strip())
            )
        except ValueError:
            # if any atom isn’t in our list, leave the original order
            chunk_sorted = chunk

        # Rewrite each line: set chain ID @ col 22, resSeq @ cols 23–26
        for L in chunk_sorted:
            if len(L) < 26:
                # too short to renumber safely: keep as-is
                new_lines.append(L)
                continue

            # build new line
            new = (
                L[:21]                          # up to chain-ID
                + chain_id                      # column 22
                + f"{res_seq:>4}"               # cols 23–26
                + L[26:]                        # rest (insertion code → end)
            )
            new_lines.append(new)

    # Write out the processed file
    with open(outfile, "w") as f:
        f.writelines(new_lines)

print("Done.")

In [ ]:
# #!/usr/bin/env python3
# import os
# import glob
# import string

# # Directory containing your PDB files
# directory = "/home/pkwon2/tmp"

# # Define the atom-ordering within each 6-line chunk
# atom_order = ["NE2", "CD2", "CG", "CB", "ND1", "CE1"]

# for infile in glob.glob(os.path.join(directory, "2ivh*com.pdb")):
#     base = os.path.basename(infile)
#     outfile = os.path.join(
#         directory,
#         base.replace(".pdb", "_chunked.pdb")
#     )
#     print(f"→ {base} → {os.path.basename(outfile)}")

#     # Read all lines
#     with open(infile, "r") as f:
#         lines = f.readlines()

#     # Split into successive groups of 6 lines
#     chunks = [lines[i : i + 6] for i in range(0, len(lines), 6)]

#     new_lines = []
#     for idx, chunk in enumerate(chunks, start=1):
#         # Letter for chain ID: A, B, C, ... (wraps around after Z)
#         chain_id = string.ascii_uppercase[(idx - 1) % len(string.ascii_uppercase)]
#         # Numeric residue sequence
#         res_seq = idx

#         # Attempt to sort by atom name (chars 14–16 → line[13:16])
#         try:
#             chunk_sorted = sorted(
#                 chunk,
#                 key=lambda L: atom_order.index(L[13:16].strip())
#             )
#         except ValueError:
#             # if any atom isn’t in our list, leave the original order
#             chunk_sorted = chunk

#         # Rewrite each line: set chain ID @ col 22, resSeq @ cols 23–26
#         for L in chunk_sorted:
#             if len(L) < 26:
#                 # too short to renumber safely: keep as-is
#                 new_lines.append(L)
#                 continue

#             # build new line
#             new = (
#                 L[:21]                          # up to chain-ID
#                 + chain_id                      # column 22
#                 + f"{res_seq:>4}"               # cols 23–26
#                 + L[26:]                        # rest (insertion code → end)
#             )
#             new_lines.append(new)

#     # Write out the processed file
#     with open(outfile, "w") as f:
#         f.writelines(new_lines)

# print("Done.")

In [ ]:
# #!/usr/bin/env python3
# """
# renumber_chunked.py

# For each '*chunked.pdb' in /home/pkwon2/tmp, replace:
#  - the atom serial number (cols 7–11) on ATOM/HETATM lines with 1,2,3…
#  - the residue sequence number (cols 23–26) on those same lines with 1,2,3…
# and overwrite the original file.
# """

# import os
# import glob
# import sys
# from pathlib import Path

# def renumber_pdb(filepath):
#     """
#     Read a PDB file, renumber ATOM/HETATM serials and residue sequence numbers,
#     and overwrite it.
#     """
#     lines = Path(filepath).read_text().splitlines(keepends=True)

#     new_lines = []
#     atom_count = 1
#     res_count  = 1

#     for line in lines:
#         if line.startswith(('ATOM  ', 'HETATM')):
#             # split into three parts:
#             #   1) record name       = cols 1–6  -> line[:6]
#             #   2) atom serial field = cols 7–11 -> new 5-char right‐justified atom_count
#             #   3) rest              = cols 12-end -> line[11:]
#             record = line[:6]
#             rest   = line[11:]

#             # build new atom serial
#             atom_serial = f"{atom_count:>5}"

#             # now, within 'rest', residue seq# lives in cols 23–26 (1-based)
#             # which are rest[11:15] (0-based slice)
#             before_res = rest[:11]
#             after_res  = rest[15:]
#             res_serial = f"{res_count:>4}"

#             # reassemble the line
#             new_line = record + atom_serial + before_res + res_serial + after_res
#             new_lines.append(new_line)

#             atom_count += 1
#             res_count  += 1
#         else:
#             new_lines.append(line)

#     Path(filepath).write_text(''.join(new_lines))
#     print(f"[+] {os.path.basename(filepath)}: "
#           f"{atom_count-1} atoms renumbered, {res_count-1} residues renumbered.")

# def main():
#     pdb_dir = '/net/scratch/pkwon2/more_tips_test_set'
#     pattern = os.path.join(pdb_dir, '*chunked.pdb')
#     files = sorted(glob.glob(pattern))

#     if not files:
#         print("No '*chunked.pdb' files found in", pdb_dir, file=sys.stderr)
#         sys.exit(1)

#     for pdb_file in files:
#         renumber_pdb(pdb_file)

# if __name__ == '__main__':
#     main()

In [8]:
#!/usr/bin/env python3
"""
renumber_and_fixchains.py

For each .pdb in /net/scratch/pkwon2/more_tips_test_set:
  - Renumber ATOM and HETATM serial (cols 7–11) from 1 upward.
  - For HETATM only, also renumber residue sequence (cols 23–26) from 1 upward.
  - For ATOM only, set chain ID (col 22) to "A".
  - Write results to renumbered/<original_name>.pdb
"""

import os
import glob

INPUT_DIR  = "/net/scratch/pkwon2/more_tips_test_set"
OUTPUT_DIR = os.path.join(INPUT_DIR, "renumbered")

def renumber_and_fixchains(in_path, out_path):
    serial_counter    = 1
    hetres_mapping    = {}
    next_het_res_num  = 1

    with open(in_path, "r") as fin, open(out_path, "w") as fout:
        for line in fin:
            if line.startswith(("ATOM  ", "HETATM")):
                # 1) Renumber atom serial (cols 7–11)
                new_serial = serial_counter
                serial_counter += 1
                new_line = line[:6] + f"{new_serial:5d}" + line[11:]

                # 2) If HETATM, renumber residue sequence (cols 23–26)
                if line.startswith("HETATM"):
                    chain_id   = line[21]                   # original
                    old_resseq = line[22:26].strip()
                    icode      = line[26]
                    key = (chain_id, old_resseq, icode)

                    if key not in hetres_mapping:
                        hetres_mapping[key] = next_het_res_num
                        next_het_res_num += 1

                    new_res = hetres_mapping[key]
                    new_line = new_line[:22] + f"{new_res:4d}" + new_line[26:]

                # 3) If ATOM, force chain ID to 'A' (col 22)
                if line.startswith("ATOM  "):
                    new_line = new_line[:21] + "A" + new_line[22:]

                fout.write(new_line)

            else:
                # leave all other lines untouched
                fout.write(line)

def main():
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    for pdb_file in glob.glob(os.path.join(INPUT_DIR, "*.pdb")):
        name = os.path.basename(pdb_file)
        out  = os.path.join(OUTPUT_DIR, name)
        print(f"Processing {name} → renumbered/{name}")
        renumber_and_fixchains(pdb_file, out)

if __name__ == "__main__":
    main()

Processing 110_0_ca_plane_ori_linesremoved_chunked.pdb → renumbered/110_0_ca_plane_ori_linesremoved_chunked.pdb
Processing 110_1_ca_plane_ori_linesremoved_chunked.pdb → renumbered/110_1_ca_plane_ori_linesremoved_chunked.pdb
Processing 110_2_ca_plane_ori_linesremoved_chunked.pdb → renumbered/110_2_ca_plane_ori_linesremoved_chunked.pdb
Processing 110_3_ca_plane_ori_linesremoved_chunked.pdb → renumbered/110_3_ca_plane_ori_linesremoved_chunked.pdb
Processing 010_1_ca_plane_ori_linesremoved_chunked.pdb → renumbered/010_1_ca_plane_ori_linesremoved_chunked.pdb
Processing 010_0_ca_plane_ori_linesremoved_chunked.pdb → renumbered/010_0_ca_plane_ori_linesremoved_chunked.pdb
Processing 010_3_ca_plane_ori_linesremoved_chunked.pdb → renumbered/010_3_ca_plane_ori_linesremoved_chunked.pdb
Processing 010_2_ca_plane_ori_linesremoved_chunked.pdb → renumbered/010_2_ca_plane_ori_linesremoved_chunked.pdb


In [11]:
#!/usr/bin/env python3
"""
generate_translated_duplicate.py

For each .pdb in /net/scratch/pkwon2/more_tips_test_set/renumbered:
  - Compute center of mass of ATOM residues A1 and A3 (chain A, resseq 1 and 3).
  - Compute translation vector = 2*(COM3 - COM1).
  - Duplicate all ATOM and HETATM lines, applying translation.
  - Assign new atom serials starting after original max serial.
  - Assign new residue sequences = old_resseq + max_original_resseq (so new protein residues start after all original residues, e.g. A13).
  - Write output with:
      * Header
      * Original ATOM lines
      * Original HETATM lines
      * Duplicated ATOM lines (translated)
      * Duplicated HETATM lines (translated)
      * Tail
  - Write results to /net/scratch/pkwon2/more_tips_test_set/renumbered_translated/<name>.pdb
"""
import os
import glob

INPUT_DIR  = "/net/scratch/pkwon2/more_tips_test_set/renumbered"

# Atomic masses for common elements
ATOM_MASSES = {
    'H': 1.008, 'C': 12.011, 'N': 14.007, 'O': 15.999, 'S': 32.06,
}

def get_atomic_mass(elem):
    return ATOM_MASSES.get(elem.strip(), 0.0)


def parse_pdb_lines(lines):
    """Split lines into header, atom_lines, and tail."""
    start = next((i for i, l in enumerate(lines) if l.startswith(("ATOM  ", "HETATM"))), len(lines))
    end = len(lines) - next((i for i, l in enumerate(reversed(lines)) if l.startswith(("ATOM  ", "HETATM"))), len(lines))
    header = lines[:start]
    atom_lines = lines[start:end]
    tail = lines[end:]
    return header, atom_lines, tail


def compute_center_of_mass(atom_lines, resseq_target):
    total = 0.0
    x_sum = y_sum = z_sum = 0.0
    for l in atom_lines:
        if not l.startswith("ATOM  "): continue
        if l[21] != 'A': continue
        if int(l[22:26]) != resseq_target: continue
        x, y, z = map(float, (l[30:38], l[38:46], l[46:54]))
        elem = l[76:78].strip()
        m = get_atomic_mass(elem)
        total += m
        x_sum += m*x; y_sum += m*y; z_sum += m*z
    if total == 0:
        raise ValueError(f"No atoms for A{resseq_target}")
    return (x_sum/total, y_sum/total, z_sum/total)


def translate_and_combine(pdb_path, out_path):
    lines = open(pdb_path).read().splitlines(keepends=True)
    header, atom_lines, tail = parse_pdb_lines(lines)

    # Centers of mass
    com1 = compute_center_of_mass(atom_lines, 1)
    com3 = compute_center_of_mass(atom_lines, 3)
    tx = 2*(com3[0] - com1[0])
    ty = 2*(com3[1] - com1[1])
    tz = 2*(com3[2] - com1[2])

    # Max serial and residue
    max_serial = max(int(l[6:11]) for l in atom_lines)
    max_resseq = max(int(l[22:26]) for l in atom_lines)
    new_serial = max_serial + 1

    with open(out_path, 'w') as fout:
        # Header
        for l in header:
            fout.write(l)
        # Original ATOM then HETATM
        for l in atom_lines:
            if l.startswith("ATOM  "):
                fout.write(l)
        for l in atom_lines:
            if l.startswith("HETATM"):
                fout.write(l)
        # Duplicated ATOM lines
        for l in atom_lines:
            if not l.startswith("ATOM  "): continue
            x, y, z = map(float, (l[30:38], l[38:46], l[46:54]))
            x += tx; y += ty; z += tz
            # Build duplicate
            dup = l[:6] + f"{new_serial:5d}" + l[11:30]
            dup += f"{x:8.3f}{y:8.3f}{z:8.3f}" + l[54:]
            new_resseq = int(l[22:26]) + max_resseq
            dup = dup[:22] + f"{new_resseq:4d}" + dup[26:]
            fout.write(dup)
            new_serial += 1
        # Duplicated HETATM lines
        for l in atom_lines:
            if not l.startswith("HETATM"): continue
            x, y, z = map(float, (l[30:38], l[38:46], l[46:54]))
            x += tx; y += ty; z += tz
            dup = l[:6] + f"{new_serial:5d}" + l[11:30]
            dup += f"{x:8.3f}{y:8.3f}{z:8.3f}" + l[54:]
            new_resseq = int(l[22:26]) + max_resseq
            dup = dup[:22] + f"{new_resseq:4d}" + dup[26:]
            fout.write(dup)
            new_serial += 1
        # Tail
        for l in tail:
            fout.write(l)


def main():
    out_dir = os.path.join(os.path.dirname(INPUT_DIR), "renumbered_translated")
    os.makedirs(out_dir, exist_ok=True)
    for pdb in glob.glob(os.path.join(INPUT_DIR, '*.pdb')):
        name = os.path.basename(pdb)
        dest = os.path.join(out_dir, name)
        print(f"Processing {name} -> {dest}")
        translate_and_combine(pdb, dest)


if __name__ == '__main__':
    main()

Processing 010_1_ca_plane_ori_linesremoved_chunked.pdb -> /net/scratch/pkwon2/more_tips_test_set/renumbered_translated/010_1_ca_plane_ori_linesremoved_chunked.pdb
Processing 010_0_ca_plane_ori_linesremoved_chunked.pdb -> /net/scratch/pkwon2/more_tips_test_set/renumbered_translated/010_0_ca_plane_ori_linesremoved_chunked.pdb
Processing 010_3_ca_plane_ori_linesremoved_chunked.pdb -> /net/scratch/pkwon2/more_tips_test_set/renumbered_translated/010_3_ca_plane_ori_linesremoved_chunked.pdb
Processing 010_2_ca_plane_ori_linesremoved_chunked.pdb -> /net/scratch/pkwon2/more_tips_test_set/renumbered_translated/010_2_ca_plane_ori_linesremoved_chunked.pdb
Processing 110_0_ca_plane_ori_linesremoved_chunked.pdb -> /net/scratch/pkwon2/more_tips_test_set/renumbered_translated/110_0_ca_plane_ori_linesremoved_chunked.pdb
Processing 110_1_ca_plane_ori_linesremoved_chunked.pdb -> /net/scratch/pkwon2/more_tips_test_set/renumbered_translated/110_1_ca_plane_ori_linesremoved_chunked.pdb
Processing 110_2_ca_pl

In [12]:
#!/usr/bin/env python3
"""
remove_ori_hetatm.py

For each PDB file in the given directory, delete any lines that start with 'HETATM'
and contain 'ORI', writing the cleaned file back in place.
"""

import glob
import os

# Directory containing renumbered/transformed PDBs
pdb_dir = "/net/scratch/pkwon2/more_tips_test_set/renumbered_translated"

# Process each .pdb file
for pdb_path in glob.glob(os.path.join(pdb_dir, "*.pdb")):
    # Read all lines
    with open(pdb_path, "r") as f:
        lines = f.readlines()

    # Filter out HETATM lines containing ORI
    filtered = [
        line for line in lines
        if not (line.startswith("HETATM") and "ORI" in line)
    ]

    # Overwrite the file with the filtered content
    with open(pdb_path, "w") as f:
        f.writelines(filtered)

    print(f"Cleaned {os.path.basename(pdb_path)} (removed {len(lines) - len(filtered)} lines)")

Cleaned 110_1_ca_plane_ori_linesremoved_chunked.pdb (removed 2 lines)
Cleaned 110_0_ca_plane_ori_linesremoved_chunked.pdb (removed 2 lines)
Cleaned 110_3_ca_plane_ori_linesremoved_chunked.pdb (removed 2 lines)
Cleaned 110_2_ca_plane_ori_linesremoved_chunked.pdb (removed 2 lines)
Cleaned 010_0_ca_plane_ori_linesremoved_chunked.pdb (removed 2 lines)
Cleaned 010_1_ca_plane_ori_linesremoved_chunked.pdb (removed 2 lines)
Cleaned 010_2_ca_plane_ori_linesremoved_chunked.pdb (removed 2 lines)
Cleaned 010_3_ca_plane_ori_linesremoved_chunked.pdb (removed 2 lines)


In [13]:
#!/usr/bin/env python3
"""
From each input PDB, produce:
  • <basename>_ca_plane_ori.pdb  – single ORI pseudo-atom offset along the minor PCA axis
Existing ORI pseudo-atoms are removed so duplicates never appear.

Minor axis is determined as the eigenvector of the calcium HETATM covariance matrix with the smallest eigenvalue; we then offset 10 Å along this vector toward glutamate oxygens (if present).
"""
import os
import glob
import numpy as np

# ── helpers ────────────────────────────────────────────────────────────────────
def compute_com(coords):
    """Compute center-of-mass of list of (x, y, z)."""
    pts = np.array(coords)
    return np.mean(pts, axis=0)


def compute_minor_axis(coords):
    """Perform PCA on coords and return eigenvector with smallest eigenvalue (unit)."""
    pts = np.array(coords)
    centroid = compute_com(pts)
    centered = pts - centroid
    cov = centered.T @ centered
    # eigen decomposition
    vals, vecs = np.linalg.eigh(cov)
    # minor axis: eigenvector with smallest eigenvalue
    minor = vecs[:, np.argmin(vals)]
    return minor / np.linalg.norm(minor)


def build_hetatm_line(x, y, z, serial=9999):
    return (f"HETATM{serial:5d}"
            f" {'ORI':>4s}"
            f" {'ORI':>3s}"
            f" {'X'}"
            f"{13:4d}"
            f"    {x:8.3f}{y:8.3f}{z:8.3f}"
            f"{1.00:6.2f}{1.00:6.2f}"
            f"{'X':>12s}")

# ── core ───────────────────────────────────────────────────────────────────────
def write_oriented_pdb(src, axis_offset=12.0):
    ca_coords, glu_coords, clean_lines = [], [], []

    with open(src) as fh:
        for ln in fh:
            # skip existing ORI lines
            if ln[12:16].strip() == 'ORI':
                continue
            clean_lines.append(ln)

            # calcium HETATM by element
            if ln.startswith('HETATM') and ln[76:78].strip() == 'Ca':
                try:
                    ca_coords.append((float(ln[30:38]), float(ln[38:46]), float(ln[46:54])))
                except ValueError:
                    pass
            # glutamate side-chain oxygens
            if ln.startswith(('ATOM','HETATM')) and ln[17:20].strip() == 'GLU' and ln[12:16].strip() in ('OE1','OE2'):
                try:
                    glu_coords.append((float(ln[30:38]), float(ln[38:46]), float(ln[46:54])))
                except ValueError:
                    pass

    if not ca_coords:
        print(f"[SKIP] {os.path.basename(src)} – no calcium HETATM found")
        return

    centroid = compute_com(ca_coords)
    minor_axis = compute_minor_axis(ca_coords)

    # orient minor axis toward glutamates if present
    if glu_coords:
        glu_com = compute_com(glu_coords)
        if np.dot(glu_com - centroid, minor_axis) < 0:
            minor_axis = -minor_axis

    # offset position along minor axis
    ori_pos = centroid + minor_axis * axis_offset
    hetatm = build_hetatm_line(*ori_pos)

    out_path = src.replace('.pdb', '_ca_plane_ori.pdb')
    with open(out_path, 'w') as out:
        out.writelines(clean_lines)
        out.write(hetatm + '\n')

    cx, cy, cz = centroid
    print(f"[WRITE] {os.path.basename(out_path)} – centroid ({cx:.3f}, {cy:.3f}, {cz:.3f}); ORI at ({ori_pos[0]:.3f}, {ori_pos[1]:.3f}, {ori_pos[2]:.3f})")


def process_pdb(pdb_path):
    write_oriented_pdb(pdb_path)

if __name__ == '__main__':
    pdb_dir = '/net/scratch/pkwon2/more_tips_test_set/renumbered_translated'
    for pdb in glob.glob(os.path.join(pdb_dir, '*.pdb')):
        process_pdb(pdb)

[WRITE] 110_1_ca_plane_ori_linesremoved_chunked_ca_plane_ori.pdb – centroid (-9.763, 4.694, -2.057); ORI at (-10.957, 4.616, -13.997)
[WRITE] 110_0_ca_plane_ori_linesremoved_chunked_ca_plane_ori.pdb – centroid (-9.782, 2.347, -2.057); ORI at (-10.970, 2.418, -13.997)
[WRITE] 110_3_ca_plane_ori_linesremoved_chunked_ca_plane_ori.pdb – centroid (-6.910, 2.347, -2.056); ORI at (-5.722, 2.418, -13.997)
[WRITE] 110_2_ca_plane_ori_linesremoved_chunked_ca_plane_ori.pdb – centroid (-6.892, 4.694, -2.057); ORI at (-5.714, 4.616, -13.998)
[WRITE] 010_0_ca_plane_ori_linesremoved_chunked_ca_plane_ori.pdb – centroid (7.442, 10.337, -1.897); ORI at (7.372, 11.465, 10.050)
[WRITE] 010_1_ca_plane_ori_linesremoved_chunked_ca_plane_ori.pdb – centroid (4.961, 10.337, -1.897); ORI at (5.031, 11.465, 10.050)
[WRITE] 010_2_ca_plane_ori_linesremoved_chunked_ca_plane_ori.pdb – centroid (7.442, 7.466, -1.897); ORI at (7.374, 6.286, 10.045)
[WRITE] 010_3_ca_plane_ori_linesremoved_chunked_ca_plane_ori.pdb – centr

In [22]:
#!/usr/bin/env python3
"""
Generate a shell script containing all the ORI‐enumeration commands
instead of executing them immediately.
"""

import glob
import os
import sys
import shlex
from shutil import which

# === Configuration ===
container_image         = "/net/software/containers/pyrosetta.sif"
enum_script             = "/projects/ml/pkwon2/enumerate_ORI_homogeneously_or_randomly_in_sphere.py"

radius                  = 4.5     # sphere radius in Å
sampling_size           = 50      # number of ORI samples
homogeneous_sampling    = True
homogeneous_surface_only = False   # only applies if homogeneous_sampling is True

stripped_dir            = "/net/scratch/pkwon2/more_tips_test_set/ori_stripped_pdbs/"
enum_output_dir         = "/net/scratch/pkwon2/more_tips_test_set/ori_outputs/"

pdb_glob                = "/net/scratch/pkwon2/more_tips_test_set/renumbered_translated/*.pdb"
commands_file           = "enum_commands.sh"

# Ensure output directories exist
os.makedirs(stripped_dir, exist_ok=True)
os.makedirs(enum_output_dir, exist_ok=True)

# Determine how to invoke the container/image
# 1) If the .sif is itself executable, call it directly
if os.path.isfile(container_image) and os.access(container_image, os.X_OK):
    engine_cmd = [container_image]
else:
    # 2) Otherwise, look for apptainer or singularity on PATH
    engine = which("apptainer") or which("singularity")
    if engine:
        engine_cmd = [engine, "exec", container_image]
    else:
        # 3) Fallback: write the image path directly (may require manual adjustment)
        engine_cmd = [container_image]
        print(
            "[WARN] No 'apptainer' or 'singularity' found in PATH; "
            "commands will start with the SIF directly.",
            file=sys.stderr
        )

def clean_pdb(in_path: str, out_path: str) -> None:
    """
    Copy all lines from in_path to out_path,
    but drop any ATOM/HETATM lines whose X/Y/Z fields cannot be parsed.
    """
    with open(in_path, "r") as inf, open(out_path, "w") as outf:
        for line in inf:
            if line.startswith(("ATOM  ", "HETATM")):
                coords = line[30:54].split()
                try:
                    _ = [float(coords[i]) for i in range(3)]
                except (ValueError, IndexError):
                    # skip malformed
                    continue
            outf.write(line)

# Gather all chunked PDBs
pdb_files = sorted(glob.glob(pdb_glob))
print(f"[DEBUG] found {len(pdb_files)} PDB(s) matching {pdb_glob}")

if not pdb_files:
    print("[ERROR] No PDBs found; check your glob path.", file=sys.stderr)
    sys.exit(1)

# Open our commands file and write a bash header
with open(commands_file, "w") as fh:
    fh.write("#!/usr/bin/env bash\n\n")
    # Loop over each PDB
    for orig_pdb in pdb_files:
        base = os.path.splitext(os.path.basename(orig_pdb))[0]
        stripped_pdb = os.path.join(stripped_dir, f"{base}_noORI.pdb")
        cleaned_pdb  = os.path.join(stripped_dir, f"{base}_noORI_cleaned.pdb")

        # 1) Strip ORI lines + capture first ORI coord
        initial_coord = None
        kept_lines = []
        with open(orig_pdb, "r") as inf:
            for line in inf:
                if line.startswith("HETATM") and line[12:16].strip() == "ORI":
                    if initial_coord is None:
                        x, y, z = line[30:38].strip(), line[38:46].strip(), line[46:54].strip()
                        initial_coord = f"{x} {y} {z}"
                    continue
                kept_lines.append(line)

        if initial_coord is None:
            print(f"[WARN] No ORI in {base}, skipping.", file=sys.stderr)
            continue

        # write stripped and cleaned PDBs
        with open(stripped_pdb, "w") as outfh:
            outfh.writelines(kept_lines)
        clean_pdb(stripped_pdb, cleaned_pdb)

        # 2) Build the command
        cmd = engine_cmd + [
            enum_script,
            "--input_pdb", cleaned_pdb,
            "--output_dir", enum_output_dir,
            "--radius", str(radius),
            "--sampling_size", str(sampling_size),
            "--initial_ORI_coord", initial_coord,
        ]
        if not homogeneous_sampling:
            cmd.append("--random")
        if homogeneous_surface_only:
            cmd.append("--homogenous_distribution_on_sphere_surface_only")

        # Format and write to file
        cmd_str = " ".join(shlex.quote(c) for c in cmd)
        fh.write(cmd_str + "\n")

print(f"Wrote {len(pdb_files)} commands to {commands_file}")

[DEBUG] found 8 PDB(s) matching /net/scratch/pkwon2/more_tips_test_set/renumbered_translated/*.pdb
Wrote 8 commands to enum_commands.sh


In [31]:
#!/usr/bin/env python3
import os
import glob

input_dir = "/net/scratch/pkwon2/more_tips_test_set/ori_outputs"
output_dir = os.path.join(input_dir, "grouped")
os.makedirs(output_dir, exist_ok=True)

def renumber_and_group_pdb(input_file, output_file):
    atom_lines = []
    hetatm_lines = []
    other_lines = []

    # Separate and filter lines
    with open(input_file, 'r') as f:
        for line in f:
            if line.startswith("ATOM"):
                atom_lines.append(line)
            elif line.startswith("HETATM"):
                hetatm_lines.append(line)
            elif not line.startswith("TER"):  # skip TER lines
                other_lines.append(line)

    # Combine ATOM first, then HETATM, then other lines
    combined = atom_lines + hetatm_lines + other_lines

    # Renumber serial numbers (columns 7-11)
    new_lines = []
    serial = 1
    for line in combined:
        if line.startswith(("ATOM", "HETATM")):
            new_line = f"{line[:6]}{serial:5d}{line[11:]}"
            serial += 1
            new_lines.append(new_line)
        else:
            new_lines.append(line)

    # Write to output
    with open(output_file, 'w') as out:
        out.writelines(new_lines)

# Process all PDB files
for pdb_file in glob.glob(os.path.join(input_dir, "*.pdb")):
    basename = os.path.basename(pdb_file)
    output_file = os.path.join(output_dir, basename)
    renumber_and_group_pdb(pdb_file, output_file)

print(f"Processed files saved to {output_dir}")

Processed files saved to /net/scratch/pkwon2/more_tips_test_set/ori_outputs/grouped


In [18]:
import os

_new_directory = "/net/scratch/pkwon2/more_tips_test_set/" ## change to correct directory

print(f"The current working directory is: {os.getcwd()}")

os.chdir(_new_directory)
print(f"Changed working directory to: {os.getcwd()}")

def create_directory_structure(base_directory):
    try:
        os.makedirs(os.path.join(base_directory, 'input'), exist_ok=True)
        os.makedirs(os.path.join(base_directory, 'outputs'), exist_ok=True)
        print(f"Directory structure '{base_directory}/input' and '{base_directory}/output' created successfully.")
    except Exception as e:
        print(f"Error creating directory structure '{base_directory}': {e}")

if __name__ == "__main__":
    base_directory = "diffusion"
    create_directory_structure(base_directory)

The current working directory is: /net/scratch/pkwon2/more_tips_test_set
Changed working directory to: /net/scratch/pkwon2/more_tips_test_set
Directory structure 'diffusion/input' and 'diffusion/output' created successfully.


In [15]:
#!/usr/bin/env python3
import os
import glob

# Paths
TASKFILE = "/net/scratch/pkwon2/more_tips_test_set/diffusion/tasks"
CKPT_DIR = "/projects/ml/pkwon2/training_repeat_protein_data/paul_yaml_from_brian2025-07-10_23:42:05.087841/rank_0/models"
PDB_DIR = "/net/scratch/pkwon2/more_tips_test_set/ori_outputs/grouped"
OUTPUT_DIR = "/net/scratch/pkwon2/more_tips_test_set/diffusion/outputs"

# Clear out any old tasks
with open(TASKFILE, "w") as f:
    pass

# List of checkpoints
pt_files = ["RFD_9.pt", "RFD_10.pt"]

# Exact string for contig_atoms
# contig_atoms = (
#     'contigmap.contig_atoms="\\\"{\'A1\':\'OE1,OE2,CD,CG\',\'A2\':\'OE1,OE2,CD,CG\','
#     '\'A3\':\'OE1,OE2,CD,CG\',\'A4\':\'OE1,OE2,CD,CG\'}\\\""'
# )

contig_atoms = (
    'contigmap.contig_atoms="\\\"{'
    '\'A1\':\'OE1,OE2,CD,CG\','
    '\'A2\':\'OE1,OE2,CD,CG\','
    '\'A3\':\'OE1,OE2,CD,CG\','
    '\'A4\':\'OE1,OE2,CD,CG\','
    '\'A14\':\'OE1,OE2,CD,CG\','
    '\'A15\':\'OE1,OE2,CD,CG\','
    '\'A16\':\'OE1,OE2,CD,CG\','
    '\'A17\':\'OE1,OE2,CD,CG\''
    '}\\\""'
)

# Loop over checkpoints and PDBs
with open(TASKFILE, "a") as f:
    for pt_file in pt_files:
        pt_path = os.path.join(CKPT_DIR, pt_file)
        pt_num = os.path.splitext(pt_file)[0].split("_")[-1]  # "8" from "RFD_8.pt"

        for pdb_file in glob.glob(os.path.join(PDB_DIR, "*.pdb")):
            basename = os.path.splitext(os.path.basename(pdb_file))[0]
            out_prefix = os.path.join(OUTPUT_DIR, f"{basename}_{pt_num}")

            cmd = (
                "/projects/ml/pkwon2/rf_diffusion_repo/rf_diffusion/run_inference.py "
                "--config-name=paul_0.yaml "
                "guidepost_bonds=True "
                "inference.align_motif=False "
                f"inference.ckpt_path={pt_path} "
                "inference.num_designs=4 "
                f"inference.output_prefix={out_prefix} "
                f"inference.input_pdb={pdb_file} "
                "inference.final_step=1 "
                "inference.contig_as_guidepost=True "
                "inference.remove_guideposts_from_output=True "
                "inference.infer_guidepost_positions=True "
                "inference.guidepost_xyz_as_design=True "
                "inference.flexible_ligand=False "
                "inference.ligand=CAL "
                "inference.state_dict_to_load=model_state_dict "
                "inference.deterministic=False "
                "inference.idealize_sidechain_outputs=True "
                "inference.radius=10.0 "
                "inference.write_trajectory=False "
                "inference.write_trb=True "
                "inference.write_trb_indep=False "
                "inference.write_trb_trajectory=False "
                "inference.center_motif=True "
                "inference.recenter_xt=False "
                "inference.update_seq_t=False "
                "inference.conditions.relative_sasa.mean=0.175 "
                "inference.conditions.relative_sasa.std=0.075 "
                "inference.conditions.radius_of_gyration.mean=12.5 "
                "inference.conditions.radius_of_gyration.std=12.5 "
                "diffuser.T=100 "
                "diffuser.aa_decode_steps=40 "
                "contigmap.contigs=['\"120-140,A1,A2,A3,A4,A14,A15,A16,A17\"'] "
                "contigmap.length='\"120-140\"' "
                f"{contig_atoms} "
                "upstream_inference_transforms.configs.SSSprinkleTransform.active=True "
                "upstream_inference_transforms.configs.SSSprinkleTransform.background=HELIX "
                "upstream_inference_transforms.configs.SSSprinkleTransform.min_mask=2 "
                "upstream_inference_transforms.configs.SSSprinkleTransform.max_mask=4 "
                "upstream_inference_transforms.configs.SSSprinkleTransform.spread_efficiency=0.7"
            )

            f.write(cmd + "\n")

In [16]:
# Define the SBATCH header, change the email to your email so it doesn't ping me!
sbatch_header = """#!/bin/bash
#SBATCH -J diffusion
#SBATCH -p cpu
#SBATCH --mem=96g
#SBATCH -t 15:00:00
#SBATCH -c 4
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=pkwon2@uw.edu

# Get the task command from the tasks file
task=$(sed -n "${SLURM_ARRAY_TASK_ID}p" tasks)

# Execute the task command
eval $task
"""

# Write the SBATCH script to a file
sbatch_file = os.path.join(base_directory, "digs_array_job.sh")
with open(sbatch_file, "w") as file:
    file.write(sbatch_header)

print(f"SBATCH script '{sbatch_file}' has been created.")

# Command to submit the job array
submit_command = "sbatch -a 1-$(cat tasks | wc -l) digs_array_job.sh"
print(f"""To submit the job array, use the following command:
cd {_new_directory}/{base_directory}\n{submit_command}""")

# Define the SBATCH header, change the email to your email so it doesn't ping me!

SBATCH script 'diffusion/digs_array_job.sh' has been created.
To submit the job array, use the following command:
cd /net/scratch/pkwon2/more_tips_test_set//diffusion
sbatch -a 1-$(cat tasks | wc -l) digs_array_job.sh


In [ ]:
#!/usr/bin/env python3
import os
import glob

# Paths
TASKFILE = "/home/pkwon2/tmp/diffusion/woody_tasks"
CKPT_DIR = "/projects/ml/pkwon2/model_checkpoints"
PDB_DIR = "/home/pkwon2/tmp/ori_outputs"
OUTPUT_DIR = "/home/pkwon2/tmp/diffusion/outputs"

# Clear out any old tasks
with open(TASKFILE, "w") as f:
    pass

# List of checkpoints
pt_files = ["RFD_140.pt"]

# Exact string for contig_atoms
contig_atoms = (
    'contigmap.contig_atoms="\\\"{\'A1\':\'OE1,OE2,CD,CG\',\'B2\':\'OE1,OE2,CD,CG\','
    '\'C3\':\'OE1,OE2,CD,CG\',\'D4\':\'OE1,OE2,CD,CG\'}\\\""'
)

# Loop over checkpoints and PDBs
with open(TASKFILE, "a") as f:
    for pt_file in pt_files:
        pt_path = os.path.join(CKPT_DIR, pt_file)
        pt_num = os.path.splitext(pt_file)[0].split("_")[-1]  # "8" from "RFD_8.pt"

        for pdb_file in glob.glob(os.path.join(PDB_DIR, "*.pdb")):
            basename = os.path.splitext(os.path.basename(pdb_file))[0]
            out_prefix = os.path.join(OUTPUT_DIR, f"{basename}_{pt_num}")

            cmd = (
                "/projects/ml/pkwon2/rf_diffusion_repo/rf_diffusion/run_inference.py "
                "--config-name=paul_0.yaml "
                "guidepost_bonds=True "
                "inference.align_motif=False "
                f"inference.ckpt_path={pt_path} "
                "inference.num_designs=3 "
                f"inference.output_prefix={out_prefix} "
                f"inference.input_pdb={pdb_file} "
                "inference.final_step=1 "
                "inference.contig_as_guidepost=True "
                "inference.remove_guideposts_from_output=True "
                "inference.infer_guidepost_positions=True "
                "inference.guidepost_xyz_as_design=True "
                "inference.flexible_ligand=False "
                "inference.ligand=CAL "
                "inference.state_dict_to_load=model_state_dict "
                "inference.deterministic=False "
                "inference.idealize_sidechain_outputs=True "
                "inference.radius=10.0 "
                "inference.write_trajectory=False "
                "inference.write_trb=True "
                "inference.write_trb_indep=False "
                "inference.write_trb_trajectory=False "
                "inference.center_motif=True "
                "inference.recenter_xt=False "
                "inference.update_seq_t=False "
                "inference.conditions.relative_sasa.mean=0.175 "
                "inference.conditions.relative_sasa.std=0.075 "
                "inference.conditions.radius_of_gyration.mean=12.5 "
                "inference.conditions.radius_of_gyration.std=12.5 "
                "diffuser.T=100 "
                "diffuser.aa_decode_steps=40 "
                "contigmap.contigs=['\"60-80,A1,B2,C3,D4\"'] "
                "contigmap.length='\"60-80\"' "
                f"{contig_atoms} "
            )

            f.write(cmd + "\n")

In [ ]:
# Define the SBATCH header, change the email to your email so it doesn't ping me!
sbatch_header = """#!/bin/bash
#SBATCH -J woody_diffusion
#SBATCH -p cpu-bf
#SBATCH --mem=48g
#SBATCH -t 05:00:00
#SBATCH -c 12
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=pkwon2@uw.edu

# Get the task command from the tasks file
task=$(sed -n "${SLURM_ARRAY_TASK_ID}p" woody_tasks)

# Execute the task command
eval $task
"""

# Write the SBATCH script to a file
sbatch_file = os.path.join(base_directory, "digs_array_job.sh")
with open(sbatch_file, "w") as file:
    file.write(sbatch_header)

print(f"SBATCH script '{sbatch_file}' has been created.")

# Command to submit the job array
submit_command = "sbatch -a 1-$(cat woody_tasks | wc -l) digs_array_job.sh"
print(f"""To submit the job array, use the following command:
cd {_new_directory}/{base_directory}\n{submit_command}""")

# Define the SBATCH header, change the email to your email so it doesn't ping me!

In [21]:
#!/usr/bin/env python3
import os
import glob
import shutil
from Bio.PDB import PDBParser, Superimposer

# === CONFIGURATION ===
ORI_DIR       = "/net/scratch/pkwon2/more_tips_test_set/ori_outputs"
INPUT_DIR     = "/net/scratch/pkwon2/more_tips_test_set/diffusion/outputs"
FILTERED_DIR  = "/net/scratch/pkwon2/more_tips_test_set/diffusion/filtered"
ATOM_NAMES    = {"OE1", "OE2", "CD", "CG"}
RMSD_CUTOFF   = 10.0  # Å

# Ensure output directory exists
os.makedirs(FILTERED_DIR, exist_ok=True)

# Initialize PDB parser and superimposer
parser = PDBParser(QUIET=True)
sup    = Superimposer()

# Process each PDB in the input directory
for pdb_path in glob.glob(os.path.join(INPUT_DIR, "*.pdb")):
    basename = os.path.basename(pdb_path)

    # Derive reference PDB name by stripping off the last two underscore-segments
    # e.g. "…ORI_72_15_3-atomized-bb-False.pdb" → "…ORI_72.pdb"
    ref_basename = basename.rsplit("_", 2)[0] + ".pdb"
    ref_path     = os.path.join(ORI_DIR, ref_basename)

    if not os.path.isfile(ref_path):
        print(f"[SKIP] {basename}: reference '{ref_basename}' not found in ori_outputs/")
        continue

    # Parse reference structure and collect reference atoms
    ref_structure = parser.get_structure("REF", ref_path)
    ref_atoms = [atom for atom in ref_structure.get_atoms()
                 if atom.get_name() in ATOM_NAMES]
    if not ref_atoms:
        print(f"[SKIP] {basename}: no atoms named {ATOM_NAMES} in {ref_basename}")
        continue

    # Parse target structure and collect target atoms
    target_structure = parser.get_structure("TGT", pdb_path)
    tgt_atoms = [atom for atom in target_structure.get_atoms()
                 if atom.get_name() in ATOM_NAMES]

    # Skip if atom counts differ
    if len(tgt_atoms) != len(ref_atoms):
        print(f"[SKIP] {basename}: atom count mismatch "
              f"(ref={len(ref_atoms)} vs tgt={len(tgt_atoms)})")
        continue

    # Compute RMSD
    sup.set_atoms(ref_atoms, tgt_atoms)
    rms = sup.rms
    print(f"{basename} → RMSD = {rms:.3f} Å")

    # Filter by cutoff
    if rms < RMSD_CUTOFF:
        dest = os.path.join(FILTERED_DIR, basename)
        shutil.copy2(pdb_path, dest)
        print(f"  → copied to filtered/")

print("Done.")

010_2_ca_plane_ori_linesremoved_chunked_ca_plane_ori_noORI_cleaned_ORI_26_9_0-atomized-bb-False.pdb → RMSD = 6.710 Å
  → copied to filtered/
110_3_ca_plane_ori_linesremoved_chunked_ca_plane_ori_noORI_cleaned_ORI_45_9_0-atomized-bb-False.pdb → RMSD = 8.570 Å
  → copied to filtered/
010_0_ca_plane_ori_linesremoved_chunked_ca_plane_ori_noORI_cleaned_ORI_70_9_0-atomized-bb-False.pdb → RMSD = 13.827 Å
010_3_ca_plane_ori_linesremoved_chunked_ca_plane_ori_noORI_cleaned_ORI_62_9_0-atomized-bb-False.pdb → RMSD = 14.120 Å
110_3_ca_plane_ori_linesremoved_chunked_ca_plane_ori_noORI_cleaned_ORI_53_9_0-atomized-bb-False.pdb → RMSD = 13.741 Å
010_0_ca_plane_ori_linesremoved_chunked_ca_plane_ori_noORI_cleaned_ORI_66_9_0-atomized-bb-False.pdb → RMSD = 9.104 Å
  → copied to filtered/
010_2_ca_plane_ori_linesremoved_chunked_ca_plane_ori_noORI_cleaned_ORI_42_9_0-atomized-bb-False.pdb → RMSD = 14.452 Å
010_1_ca_plane_ori_linesremoved_chunked_ca_plane_ori_noORI_cleaned_ORI_08_9_0-atomized-bb-False.pdb → RMS

In [22]:
# Cell 7: Enhanced clash filtering based on all protein atom ↔ HETATM distances

import os
import shutil
import numpy as np
import pandas as pd
from Bio.PDB import PDBParser

# Parameters
clash_threshold = 2.0  # Å; any protein–HETATM pair closer than this is a clash
source_dir   = "/net/scratch/pkwon2/more_tips_test_set/diffusion/filtered"
dest_dir     = "/net/scratch/pkwon2/more_tips_test_set/diffusion/clash_filtered"
os.makedirs(dest_dir, exist_ok=True)

# Prepare parser and file list
parser    = PDBParser(QUIET=True)
pdb_paths = [os.path.join(source_dir, fn)
             for fn in os.listdir(source_dir) if fn.endswith(".pdb")]

# Collect clash statistics
clash_stats = []

for path in pdb_paths:
    name      = os.path.basename(path)
    struct    = parser.get_structure(name, path)
    
    # extract protein atoms (standard residues) and HETATM atoms
    prot_atoms = [
        atom.get_coord()
        for model in struct
        for chain in model
        for residue in chain
        if residue.id[0] == ' '  # standard amino acid
        for atom in residue
    ]
    het_atoms = [
        atom.get_coord()
        for model in struct
        for chain in model
        for residue in chain
        if residue.id[0] != ' '  # HETATM
        for atom in residue
    ]
    
    # if no hetatoms, treat as zero clashes
    if not het_atoms:
        clash_count = 0
    else:
        prot_coords = np.array(prot_atoms)
        het_coords  = np.array(het_atoms)
        # compute all pairwise distances
        dists = np.linalg.norm(prot_coords[:, None, :] - het_coords[None, :, :], axis=2)
        clash_count = int(np.sum(dists < clash_threshold))
    
    clash_stats.append({'pdb_file': name, 'clashes': clash_count})
    
    # copy only those with zero clashes
    if clash_count == 0:
        shutil.copy2(path, os.path.join(dest_dir, name))

# write summary to CSV
clash_df = pd.DataFrame(clash_stats)
clash_df.to_csv("/net/scratch/pkwon2/more_tips_test_set/diffusion/clash_summary.csv", index=False)

print(f"Filtered complete: {len(clash_df[clash_df['clashes']==0])} files passed (0 clashes) and were copied to\n  {dest_dir}")

Filtered complete: 17 files passed (0 clashes) and were copied to
  /net/scratch/pkwon2/more_tips_test_set/diffusion/clash_filtered


In [ ]:
#!/usr/bin/env python3
import os
import collections
import matplotlib.pyplot as plt

# — CONFIG —
FILTERED_DIR = "/home/pkwon2/tmp/diffusion/clash_filtered"
# — END CONFIG —

def main():
    # 1) Gather counts by the “110_3”-style prefix
    counts = collections.Counter()
    for fname in os.listdir(FILTERED_DIR):
        if not fname.endswith(".pdb"):
            continue

        # take the first two underscore-separated fields
        # e.g. "110_3_ca_plane_....pdb" → ["110","3", ...] → "110_3"
        parts = fname.split("_", 2)
        if len(parts) < 2:
            print(f"[WARN] filename too short to group: {fname}")
            continue
        group = "_".join(parts[:2])
        counts[group] += 1

    if not counts:
        print("No PDB files found in filtered directory.")
        return

    # 2) Sort groups numerically by both parts
    def sort_key(name):
        try:
            a, b = name.split("_")
            return (int(a), int(b))
        except ValueError:
            return (name, 0)

    groups = sorted(counts.keys(), key=sort_key)
    values = [counts[g] for g in groups]

    # 3) Plot
    plt.figure(figsize=(8, 5))
    plt.bar(groups, values)
    plt.xticks(rotation=45, ha="right")
    plt.xlabel("Group (e.g. 110_3)")
    plt.ylabel("Number of filtered PDBs")
    plt.title("Filtered PDB Count per 110_x Group")
    plt.tight_layout()
    plt.show()

if __name__ == "__main__":
    main()

In [ ]:
# #!/usr/bin/env python3
# import os, collections, shutil
# import numpy as np
# from Bio.PDB import PDBParser
# from scipy.spatial import cKDTree
# import matplotlib.pyplot as plt

# # — CONFIG —
# FILTERED_DIR = "/home/pkwon2/tmp/diffusion/filtered"
# UNCLASH_DIR  = "/home/pkwon2/tmp/diffusion/unclashing"
# CLASH_DIR    = "/home/pkwon2/tmp/diffusion/clashing"
# CLASH_DIST   = 0.8   # Å
# # — END CONFIG —

# os.makedirs(UNCLASH_DIR, exist_ok=True)
# os.makedirs(CLASH_DIR,   exist_ok=True)

# parser = PDBParser(QUIET=True)

# def is_unclashing(pdb_path):
#     """Return True if no two atoms (ATOM+HETATM) are within CLASH_DIST."""
#     struct = parser.get_structure("X", pdb_path)
#     coords = np.array([atom.coord for atom in struct.get_atoms()])
#     if coords.shape[0] < 2:
#         return False
#     tree = cKDTree(coords)
#     # query_pairs returns all (i,j) with dist <= CLASH_DIST
#     clashes = tree.query_pairs(r=CLASH_DIST)
#     return len(clashes) == 0

# def main():
#     counts = collections.Counter()

#     for fname in os.listdir(FILTERED_DIR):
#         if not fname.endswith(".pdb"):
#             continue
#         src = os.path.join(FILTERED_DIR, fname)
#         if is_unclashing(src):
#             shutil.copy2(src, os.path.join(UNCLASH_DIR, fname))
#             grp = "_".join(fname.split("_", 2)[:2])
#             counts[grp] += 1
#         else:
#             shutil.copy2(src, os.path.join(CLASH_DIR, fname))

#     if not counts:
#         print("No unclashing PDBs found.")
#         return

#     # plot
#     groups = sorted(counts, key=lambda g: tuple(map(int, g.split("_"))))
#     values = [counts[g] for g in groups]
#     plt.figure(figsize=(8,5))
#     plt.bar(groups, values)
#     plt.xticks(rotation=45, ha="right")
#     plt.xlabel("Group (e.g. 110_3)")
#     plt.ylabel("Unclashing PDB Count")
#     plt.title("Unclashing per 110_x (using cKDTree)")
#     plt.tight_layout()
#     plt.show()

# if __name__ == "__main__":
#     main()

In [ ]:
base_directory = "mpnn"
create_directory_structure(base_directory)

In [ ]:
#!/usr/bin/env python3
import os
from Bio import PDB

def find_non_charged_residues(file_path):
    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure('PDB_structure', file_path)
    residue_positions = []
    charged_residues = ['GLU']

    for model in structure:
        for chain in model:
            residues_list = list(chain.get_residues())
            if len(residues_list) > 1:
                residues_to_process = residues_list[:]
            else:
                residues_to_process = []

            for residue in residues_to_process:
                resname = residue.get_resname()
                residue_number = residue.id[1]

                if resname not in charged_residues:
                    residue_positions.append(f"{chain.id}{residue_number}")
    return ' '.join(residue_positions)

def find_fixed_residues(file_path):
    parser = PDB.PDBParser(QUIET=True)
    structure = parser.get_structure('PDB_structure', file_path)
    fixed_positions = []

    for model in structure:
        for chain in model:
            for residue in chain.get_residues():
                if residue.get_resname() == 'GLU':
                    residue_number = residue.id[1]
                    fixed_positions.append(f"{chain.id}{residue_number}")
    return ' '.join(fixed_positions)

def main():
    pdb_dir = './diffusion/clash_filtered'
    pdb_dir = os.path.abspath(pdb_dir)

    if not os.path.exists(pdb_dir):
        print(f"Directory {pdb_dir} does not exist.")
        return

    base_directory = 'mpnn'
    os.makedirs(base_directory, exist_ok=True)

    base_command = (
        "/software/containers/mlfold.sif /databases/mpnn/fused_mpnn/run.py "
        "--model_type 'ligand_mpnn' "
        "--temperature 0.3 "
        "--redesigned_residues '{residues}' "
        "--ligand_mpnn_use_side_chain_context 1 "
        "--fixed_residues '{fixed_residues}' "
        "--pdb_path {file_path} "
        "--out_folder outputs "
        "--batch_size 2 "
        "--number_of_batches 2 "
        "--bias_AA 'M:-10.0' "
        "--pack_side_chains 1 "
        "--repack_everything 0"
    )

    tasks_file = os.path.join(base_directory, 'tasks')

    with open(tasks_file, 'w') as file:
        for pdb_file in os.listdir(pdb_dir):
            if pdb_file.endswith('.pdb'):
                file_path = os.path.join(pdb_dir, pdb_file)
                residues = find_non_charged_residues(file_path)
                fixed_residues = find_fixed_residues(file_path)
                command = base_command.format(
                    file_path=file_path,
                    residues=residues,
                    fixed_residues=fixed_residues
                )
                file.write(f"{command}\n")

    print(f"Tasks file '{tasks_file}' has been created.")

if __name__ == '__main__':
    main()

In [ ]:
# Define the SBATCH header, change the email to your email so it doesn't ping me!
sbatch_header = """#!/bin/bash
#SBATCH -J mpnn
#SBATCH -p cpu
#SBATCH --cpus-per-task=4
#SBATCH -t 04:00:00
#SBATCH --mem=16g
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=pkwon2@uw.edu

# Get the task command from the tasks file
task=$(sed -n "${SLURM_ARRAY_TASK_ID}p" tasks)

# Execute the task command
eval $task
"""

# Write the SBATCH script to a file
sbatch_file = os.path.join(base_directory, "digs_array_job.sh")
with open(sbatch_file, "w") as file:
    file.write(sbatch_header)

print(f"SBATCH script '{sbatch_file}' has been created.")

# Command to submit the job array
submit_command = "sbatch -a 1-$(cat tasks | wc -l) digs_array_job.sh"
print(f"""To submit the job array, use the following command:
cd {_new_directory}/{base_directory}\n{submit_command}""")

In [ ]:
base_directory = "superfold"
create_directory_structure(base_directory)

def main():
    pdb_dir = './mpnn/outputs/packed' #this is where your mpnn outputs are
    
    # Convert relative path to absolute path
    pdb_dir = os.path.abspath(pdb_dir)

    if not os.path.exists(pdb_dir):
        print(f"Directory {pdb_dir} does not exist.")
        return

    # Define the base command
    base_command = "/net/software/lab/superfold/pyroml/bin/python3.8 /software/lab/superfold/run_superfold.py {file_path} " \
                   "--models 3 " \
                   "--initial_guess " \
                   "--out_dir outputs " \

    tasks_file = os.path.join(base_directory, "tasks")

    with open(tasks_file, "w") as file:
        for pdb_file in os.listdir(pdb_dir):
            if pdb_file.endswith('.pdb'):
                file_path = os.path.join(pdb_dir, pdb_file)
                command = base_command.format(file_path=file_path)
                file.write(f"{command}\n")

    print(f"Tasks file '{tasks_file}' has been created.")

if __name__ == "__main__":
    main()

In [ ]:
# Define the SBATCH header, change the email to your email so it doesn't ping me!
sbatch_header = """#!/bin/bash
#SBATCH -J af2
#SBATCH -p cpu-bf
#SBATCH --mem=48g
#SBATCH -t 05:00:00
#SBATCH -c 12
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=pkwon2@uw.edu

# Get the task command from the tasks file
task=$(sed -n "${SLURM_ARRAY_TASK_ID}p" tasks)

# Execute the task command
eval $task
"""

# Write the SBATCH script to a file
sbatch_file = os.path.join(base_directory, "digs_array_job.sh")
with open(sbatch_file, "w") as file:
    file.write(sbatch_header)

print(f"SBATCH script '{sbatch_file}' has been created.")

# Command to submit the job array
submit_command = "sbatch -a 1-$(cat tasks | wc -l) digs_array_job.sh"
print(f"""To submit the job array, use the following command:
cd {_new_directory}/{base_directory}\n{submit_command}""")

In [ ]:
base_directory = "superfold"
create_directory_structure(base_directory)

import os
import pandas as pd
import numpy as np
from Bio.PDB import PDBParser, Superimposer
import shutil

# Define directories
packed_dir = './mpnn/outputs/packed'
superfold_outputs_dir = './superfold/outputs'
rmsd_filtered_dir = './superfold/rmsd_filtered'

# Create the filtered directory if it doesn't exist
os.makedirs(rmsd_filtered_dir, exist_ok=True)

# Initialize PDB parser
parser = PDBParser(QUIET=True)

def calculate_rmsd(file1, file2):
    """Calculate RMSD between two PDB files using CA atoms."""
    try:
        structure1 = parser.get_structure('structure1', file1)
        structure2 = parser.get_structure('structure2', file2)

        # Extract CA atoms and order by residue number
        atoms1 = []
        atoms2 = []
        for model1, model2 in zip(structure1, structure2):
            for chain1, chain2 in zip(model1, model2):
                residues1 = [res for res in chain1 if res.has_id('CA')]
                residues2 = [res for res in chain2 if res.has_id('CA')]

                if len(residues1) != len(residues2):
                    print(f'Number of CA residues differ between {file1} and {file2}')
                    return None

                for res1, res2 in zip(residues1, residues2):
                    atoms1.append(res1['CA'])
                    atoms2.append(res2['CA'])

        if len(atoms1) != len(atoms2):
            print(f'Number of CA atoms differ between {file1} and {file2}')
            return None

        # Superimpose structures
        super_imposer = Superimposer()
        super_imposer.set_atoms(atoms1, atoms2)
        super_imposer.apply(structure2.get_atoms())

        return super_imposer.rms
    except Exception as e:
        print(f'Error calculating RMSD between {file1} and {file2}: {e}')
        return None

# Dataframe to store paths and RMSD values
data = []

# Iterate over files in the packed directory
for packed_pdb in os.listdir(packed_dir):
    if packed_pdb.endswith('.pdb'):
        packed_pdb_path = os.path.join(packed_dir, packed_pdb)
        # Loop over models 1 to 5
        for model_num in range(3, 4):
            # Construct corresponding file names in superfold directory
            superfold_pdb = packed_pdb.replace('.pdb', f'_model_{model_num}_ptm_seed_0_unrelaxed.pdb')
            print(f'Processing: {superfold_pdb}')
            superfold_json = superfold_pdb.replace('unrelaxed.pdb', 'prediction_results.json')
            
            superfold_pdb_path = os.path.join(superfold_outputs_dir, superfold_pdb)
            superfold_json_path = os.path.join(superfold_outputs_dir, superfold_json)
            
            # Ensure the corresponding superfold pdb and json files exist
            if os.path.exists(superfold_pdb_path) and os.path.exists(superfold_json_path):
                # Calculate RMSD
                rmsd = calculate_rmsd(packed_pdb_path, superfold_pdb_path)
                
                if rmsd is not None:
                    # Check RMSD threshold
                    if rmsd < 2:
                        # Copy pdb and json files to rmsd_filtered directory
                        shutil.copy(superfold_pdb_path, rmsd_filtered_dir)
                        shutil.copy(superfold_json_path, rmsd_filtered_dir)
                        
                        # Add to dataframe
                        data.append([packed_pdb_path, superfold_pdb_path, rmsd])
                else:
                    print(f'Skipping due to RMSD calculation error between {packed_pdb_path} and {superfold_pdb_path}')
            else:
                print(f'Missing files for {superfold_pdb_path} or {superfold_json_path}')

# Convert to DataFrame and save to CSV
df = pd.DataFrame(data, columns=['Packed PDB Path', 'Superfold PDB Path', 'RMSD'])
csv_path = os.path.join(rmsd_filtered_dir, 'rmsd_filtered.csv')
df.to_csv(csv_path, index=False)

print(f'RMSD calculation and file filtering complete. Results saved to {csv_path}')

In [ ]:
import os
import json

# Define the directory containing the .json and .pdb files
directory = './superfold/rmsd_filtered'

# Function to delete a file
def delete_file(file_path):
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f'Deleted {file_path}')
    else:
        print(f'{file_path} does not exist')

# Loop through all files in the directory
for filename in os.listdir(directory):
    if filename.endswith('prediction_results.json'):
        json_path = os.path.join(directory, filename)
        
        try:
            with open(json_path, 'r') as f:
                data = json.load(f)
            
            mean_plddt = data.get('mean_plddt', 0)
            mean_pae = data.get('mean_pae', 999)
            
            # Delete if criteria are not met
            if mean_plddt <= 80 or mean_pae >= 5:
                delete_file(json_path)
                
                # Delete associated .pdb file
                pdb_filename = filename.replace('prediction_results.json', 'unrelaxed.pdb')
                pdb_path = os.path.join(directory, pdb_filename)
                delete_file(pdb_path)

        except (json.JSONDecodeError, IOError) as e:
            print(f"Error reading {json_path}: {e}")

print('Filtering complete.')

In [ ]:
import os
import json
from collections import defaultdict

# Define the directory containing the .json and .pdb files
directory = './superfold/rmsd_filtered'

# Initialize a dictionary to hold groups of files
file_groups = defaultdict(list)

# Function to extract the prefix based on the last instance of 'packed'
def get_prefix(filename):
    parts = filename.split('packed')
    if len(parts) > 1:
        return 'packed'.join(parts[:-1]) + 'packed'
    return filename

# Loop through all files in the directory
for filename in os.listdir(directory):
    if filename.endswith('prediction_results.json'):
        prefix = get_prefix(filename)
        json_path = os.path.join(directory, filename)
        
        # Read the .json file
        with open(json_path, 'r') as json_file:
            data = json.load(json_file)
            
            # Get the mean_pae value; default to a high value if not found so it doesn't win
            mean_pae = data.get('mean_pae', 999)
            
            # Append the file info to the group
            file_groups[prefix].append((mean_pae, json_path, filename))

# Function to delete a file
def delete_file(file_path):
    if os.path.exists(file_path):
        os.remove(file_path)
        print(f'Deleted {file_path}')
    else:
        print(f'{file_path} does not exist')

# Iterate through each group
for prefix, files in file_groups.items():
    # Sort the files by mean_pae in ascending order (lowest value first)
    files.sort(key=lambda x: x[0])
    
    # Get the lowest n files (modify 'n' to choose lowest 1, 2, 3, etc.)
    n = 1 # For the lowest 1 file per group; change to 3 for the lowest 3 per group
    lowest_files = files[:n]
    
    # Files to keep
    lowest_files_set = set(f[1] for f in lowest_files)
    
    # Iterate through the files in the group and delete those not selected
    for _, json_path, filename in files:
        if json_path not in lowest_files_set:
            # Delete the .json file
            delete_file(json_path)
            
            # Construct the corresponding .pdb filename and delete it
            pdb_filename = filename.replace('prediction_results.json', 'unrelaxed.pdb')
            pdb_path = os.path.join(directory, pdb_filename)
            delete_file(pdb_path)

print('Processing complete.')

In [ ]:
#!/usr/bin/env python3
import re
import shutil
from pathlib import Path

# === Configuration ===
SUPERFOLD_DIR = Path("/home/pkwon2/tmp/superfold/rmsd_filtered")
DIFFUSION_DIR = Path("/home/pkwon2/tmp/diffusion/filtered")
OUTPUT_DIR    = Path("/home/pkwon2/tmp/superfold/rmsd_filtered/paired_filtered")

# Ensure output directory exists
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Precompute all diffusion stems for quick membership checks
diffusion_bases = {p.stem for p in DIFFUSION_DIR.glob("*.pdb")}
paired = set()

# Iterate through superfold files and attempt pairing
for sf_file in SUPERFOLD_DIR.glob("*.pdb"):
    stem = sf_file.stem
    # Extract everything up to "_packed" (if present), otherwise whole stem
    m = re.match(r"(.+?)_packed", stem)
    key = m.group(1) if m else stem

    diffusion_path = DIFFUSION_DIR / f"{key}.pdb"
    print(f"Looking for: {diffusion_path.name}")

    if key in diffusion_bases:
        # Copy both PDBs into the output directory
        shutil.copy(sf_file, OUTPUT_DIR / sf_file.name)
        shutil.copy(diffusion_path, OUTPUT_DIR / diffusion_path.name)
        paired.add(key)
        print(f"Paired: {sf_file.name} <--> {diffusion_path.name}")
    else:
        print(f"[WARNING] No diffusion match for: {sf_file.name}")

# Report any diffusion files that never found a superfold partner
for leftover in sorted(diffusion_bases - paired):
    print(f"[INFO] Diffusion-only (no Superfold): {leftover}.pdb")

print("Pairing complete.")

In [ ]:
import os
import glob
import re
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

# Directory containing filtered PDB files
pdb_dir = "/home/pkwon2/tmp/superfold/rmsd_filtered"
pdb_files = glob.glob(os.path.join(pdb_dir, "*.pdb"))

# Data structure: { group -> { subgroup -> count } }
group_counts = defaultdict(lambda: defaultdict(int))
all_subgroups = set()

for path in pdb_files:
    filename = os.path.basename(path)

    # Extract group (first two parts separated by _)
    parts = filename.split("_")
    if len(parts) < 2:
        continue
    group = f"{parts[0]}_{parts[1]}"

    # Extract subgroup from the ORI segment
    match = re.search(r"ORI_\d+_(\d+)_", filename)
    if match:
        subgroup = match.group(1)
        group_counts[group][subgroup] += 1
        all_subgroups.add(subgroup)
    else:
        print(f"Could not parse ORI subgroup from {filename}")

# Sort subgroup labels numerically
sorted_subgroups = sorted(all_subgroups, key=int)
x = np.arange(len(sorted_subgroups))  # the label locations

# Plotting setup
width = 0.8 / len(group_counts)  # bar width
fig, ax = plt.subplots(figsize=(16, 6))

# Assign a unique color to each group
colors = plt.cm.tab20.colors
group_color_map = {group: colors[i % len(colors)] for i, group in enumerate(group_counts)}

# Plot each group's counts
for i, (group, subgroup_dict) in enumerate(group_counts.items()):
    counts = [subgroup_dict.get(sub, 0) for sub in sorted_subgroups]
    offset = (i - len(group_counts)/2) * width + width/2
    ax.bar(x + offset, counts, width, label=group, color=group_color_map[group])

# Final touches
ax.set_xlabel("Checkpoint")
ax.set_ylabel("Number of backbones")
ax.set_title("Average number of unique backbones with different weights")
ax.set_xticks(x)
ax.set_xticklabels(sorted_subgroups, rotation=45)
ax.legend(title="Facet Task")
plt.tight_layout()
plt.savefig("aggregate_grouped_subgroup_distribution.png")
print("Saved plot: aggregate_grouped_subgroup_distribution.png")
plt.close()

In [ ]:
import os
import glob
import re
import matplotlib.pyplot as plt
import numpy as np
from collections import defaultdict

# Directory containing filtered PDB files
pdb_dir = "/home/pkwon2/tmp/superfold/rmsd_filtered"
pdb_files = glob.glob(os.path.join(pdb_dir, "*.pdb"))

# Data structure: { group -> { subgroup -> count } }
group_counts = defaultdict(lambda: defaultdict(int))
all_subgroups = set()

for path in pdb_files:
    filename = os.path.basename(path)

    # Extract group (first two parts separated by _)
    parts = filename.split("_")
    if len(parts) < 2:
        continue
    group = f"{parts[0]}_{parts[1]}"

    # Extract subgroup from the ORI segment
    match = re.search(r"ORI_\d+_(\d+)_", filename)
    if match:
        subgroup = match.group(1)
        group_counts[group][subgroup] += 1
        all_subgroups.add(subgroup)
    else:
        print(f"Could not parse ORI subgroup from {filename}")

# Sort subgroup labels numerically
sorted_subgroups = sorted(all_subgroups, key=int)
x = np.arange(len(sorted_subgroups))  # the label locations

# =========================
# 1. Grouped Bar Plot
# =========================
width = 0.8 / len(group_counts)  # bar width
fig, ax = plt.subplots(figsize=(16, 6))

# Assign a unique color to each group
colors = plt.cm.tab20.colors
group_color_map = {group: colors[i % len(colors)] for i, group in enumerate(group_counts)}

# Plot each group's counts
for i, (group, subgroup_dict) in enumerate(group_counts.items()):
    counts = [subgroup_dict.get(sub, 0) for sub in sorted_subgroups]
    offset = (i - len(group_counts)/2) * width + width/2
    ax.bar(x + offset, counts, width, label=group, color=group_color_map[group])

# Final touches for grouped plot
ax.set_xlabel("Checkpoint")
ax.set_ylabel("Number of backbones")
ax.set_title("Unique backbones per task with different weights")
ax.set_xticks(x)
ax.set_xticklabels(sorted_subgroups, rotation=45)
ax.legend(title="Facet Task")
plt.tight_layout()
plt.savefig("aggregate_grouped_subgroup_distribution.png")
print("Saved plot: aggregate_grouped_subgroup_distribution.png")
plt.close()

# =========================
# 2. Average ± Std Plot
# =========================
# Build a matrix: rows = groups, cols = sorted_subgroups
subgroup_matrix = []
for group in group_counts:
    row = [group_counts[group].get(sub, 0) for sub in sorted_subgroups]
    subgroup_matrix.append(row)

subgroup_array = np.array(subgroup_matrix)
means = np.mean(subgroup_array, axis=0)
stds = np.std(subgroup_array, axis=0)

# Plot average ± std
fig, ax = plt.subplots(figsize=(16, 6))
ax.bar(x, means, yerr=stds, capsize=5, color="skyblue", edgecolor="black")
ax.set_xlabel("Subgroup (from ORI_X_Y_Z)")
ax.set_ylabel("Average Count ± Std")
ax.set_title("Average Count per Subgroup with Standard Deviation")
ax.set_xticks(x)
ax.set_xticklabels(sorted_subgroups, rotation=45)
plt.tight_layout()
plt.savefig("subgroup_mean_std_plot.png")
print("Saved plot: subgroup_mean_std_plot.png")
plt.close()

In [ ]:
#!/usr/bin/env python3
import re
import shutil
from pathlib import Path

# === Configuration ===
SUPERFOLD_DIR = Path("/home/pkwon2/tmp/superfold/rmsd_filtered")
DIFFUSION_DIR = Path("/home/pkwon2/tmp/diffusion/filtered")
OUTPUT_DIR    = Path("/home/pkwon2/tmp/superfold/rmsd_filtered/paired_filtered")

# Ensure output directory exists
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Precompute all diffusion stems for quick membership checks
diffusion_bases = {p.stem for p in DIFFUSION_DIR.glob("*.pdb")}
paired = set()

# Iterate through superfold files and attempt pairing
for sf_file in SUPERFOLD_DIR.glob("*.pdb"):
    stem = sf_file.stem
    # Extract everything up to "_packed" (if present), otherwise whole stem
    m = re.match(r"(.+?)_packed", stem)
    key = m.group(1) if m else stem

    diffusion_path = DIFFUSION_DIR / f"{key}.pdb"
    print(f"Looking for: {diffusion_path.name}")

    if key in diffusion_bases:
        # Copy both PDBs into the output directory
        shutil.copy(sf_file, OUTPUT_DIR / sf_file.name)
        shutil.copy(diffusion_path, OUTPUT_DIR / diffusion_path.name)
        paired.add(key)
        print(f"Paired: {sf_file.name} <--> {diffusion_path.name}")
    else:
        print(f"[WARNING] No diffusion match for: {sf_file.name}")

# Report any diffusion files that never found a superfold partner
for leftover in sorted(diffusion_bases - paired):
    print(f"[INFO] Diffusion-only (no Superfold): {leftover}.pdb")

print("Pairing complete.")

In [ ]:
#!/usr/bin/env python3
import os
import shutil
from pathlib import Path

# === Configuration ===
SUPERFOLD_DIR = Path("/home/pkwon2/tmp/superfold/rmsd_filtered/paired_filtered")
DIFFUSION_DIR = Path("/home/pkwon2/tmp/ori_outputs")
OUTPUT_DIR    = Path("/home/pkwon2/tmp/diffusion/good_ori")

# Ensure output directory exists
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Iterate through superfold files
for sf_file in SUPERFOLD_DIR.glob("*False.pdb"):
    # Extract base key from superfold filename
    parts = sf_file.stem.split("_15_")[0]  # remove suffix like "_model_3_ptm_seed_0_unrelaxed"
    diffusion_file = DIFFUSION_DIR / f"{parts}.pdb"

    # Check if corresponding diffusion file exists
    if diffusion_file.exists():
        # Copy both files to output directory
        shutil.copy(sf_file, OUTPUT_DIR / sf_file.name)
        shutil.copy(diffusion_file, OUTPUT_DIR / diffusion_file.name)
        print(f"Paired: {sf_file.name} <--> {diffusion_file.name}")
    else:
        print(f"[WARNING] No match for: {sf_file.name}")

print("Pairing complete.")

In [ ]:
#!/usr/bin/env python3
import os
import glob

# ——————————————————————————————————————————————————————————————
# User-configurable paths and lists
# ——————————————————————————————————————————————————————————————
configs_dir = "/home/pkwon2/tmp/configs"
base_ckpt_dir = (
    "/net/scratch/pkwon2/training_repeat_protein_data/"
    "paul_yaml_from_woody_rfd1402025-04-29_23:13:20.252396/"
    "rank_0/models"
)
# Automatically gather all PDBs in the enumeration output directory
pdb_dir = "/home/pkwon2/tmp/diffusion/good_ori"

# ——————————————————————————————————————————————————————————————
# Prepare output directory
# ——————————————————————————————————————————————————————————————
os.makedirs(configs_dir, exist_ok=True)

# ——————————————————————————————————————————————————————————————
# Loop and write YAMLs
# ——————————————————————————————————————————————————————————————
# Find all .pdb files in pdb_dir
pdb_files = sorted(glob.glob(os.path.join(pdb_dir, "*.pdb")))
if not pdb_files:
    print(f"No PDB files found in {pdb_dir}")
    exit(1)

# Define checkpoints to iterate
checkpoints = [
    "RFD_15.pt",
]

for pdb_path in pdb_files:
    # Derive prefix name (basename without extension)
    base_name = os.path.basename(pdb_path)
    pdb_prefix = os.path.splitext(base_name)[0]

    for ckpt in checkpoints:
        ckpt_path = os.path.join(base_ckpt_dir, ckpt)
        # Extract numeric part, e.g. "15" from "RFD_15.pt"
        ckpt_num = ckpt.split("_")[1].split(".")[0]
        output_prefix = f"/home/pkwon2/tmp/diffusion/outputs/{pdb_prefix}_{ckpt_num}"
        fname = f"fd31_{pdb_prefix}_{ckpt_num}.yaml"
        outpath = os.path.join(configs_dir, fname)

        # YAML content
        yaml_text = f"""\
defaults:
  - aa_v2

guidepost_bonds: True

inference:
  align_motif: False
  ckpt_path: {ckpt_path}
  num_designs: 30
  output_prefix: {output_prefix}
  input_pdb: {pdb_path}
  final_step: 1
  contig_as_guidepost: True
  remove_guideposts_from_output: True
  infer_guidepost_positions: True
  guidepost_xyz_as_design: True
  flexible_ligand: False
  ligand: CAL
  state_dict_to_load: model_state_dict
  deterministic: False
  idealize_sidechain_outputs: True
  radius: 10.0
  write_trajectory: False
  write_trb: True
  write_trb_indep: False
  write_trb_trajectory: False
  center_motif: True
  recenter_xt: False
  update_seq_t: False
  conditions:
    relative_sasa:
      mean: 0.175
      std: 0.075
    radius_of_gyration:
      mean: 12.5
      std: 12.5

contigmap:
  contigs: ['60-80,A1,B2,C3,D4']
  length: '60-80'
  contig_atoms: '{{"A1":"OE1,OE2,CD,CG","B2":"OE1,OE2,CD,CG","C3":"OE1,OE2,CD,CG","D4":"OE1,OE2,CD,CG"}}'

diffuser:
  T: 100
  aa_decode_steps: 40
"""
        with open(outpath, "w") as f:
            f.write(yaml_text)

        print(f"✅ Wrote {outpath}")

In [ ]:
#!/usr/bin/env python3
import os
import glob

# ——————————————————————————————————————————————————————————————
# Paths
# ——————————————————————————————————————————————————————————————
config_dir = "/home/pkwon2/tmp/configs"
tasks_file = "/home/pkwon2/tmp/diffusion/tasks"
runner = "/home/pkwon2/rf_diffusion/rf_diffusion_repo/rf_diffusion/run_inference.py"

# Ensure the diffusion directory exists
os.makedirs(os.path.dirname(tasks_file), exist_ok=True)

# Gather all YAML configs
config_paths = sorted(glob.glob(os.path.join(config_dir, "*.yaml")))

# Write one task line per config
with open(tasks_file, "w") as out:
    for cfg in config_paths:
        name = os.path.splitext(os.path.basename(cfg))[0]
        line = (
            f"{runner} "
            f"--config-dir={config_dir} "
            f"--config-name={name}"
        )
        out.write(line + "\n")

print(f"Wrote {len(config_paths)} tasks to {tasks_file}")

In [ ]:
# Define the SBATCH header, change the email to your email so it doesn't ping me!
sbatch_header = """#!/bin/bash
#SBATCH -J diffusion
#SBATCH -p gpu-bf
#SBATCH --mem=48g
#SBATCH --exclude=gpu[3-24],gpu[71-83],gpu[110-141],gpu[51-60]
#SBATCH -t 03:00:00
#SBATCH -c 12
#SBATCH --gres=gpu
#SBATCH --mail-type=END,FAIL
#SBATCH --mail-user=pkwon2@uw.edu

# Get the task command from the tasks file
task=$(sed -n "${SLURM_ARRAY_TASK_ID}p" tasks)

# Execute the task command
eval $task
"""

# Write the SBATCH script to a file
sbatch_file = os.path.join(base_directory, "digs_array_job.sh")
with open(sbatch_file, "w") as file:
    file.write(sbatch_header)

print(f"SBATCH script '{sbatch_file}' has been created.")

# Command to submit the job array
submit_command = "sbatch -a 1-$(cat tasks | wc -l) digs_array_job.sh"
print(f"""To submit the job array, use the following command:
cd {_new_directory}/{base_directory}\n{submit_command}""")

# Define the SBATCH header, change the email to your email so it doesn't ping me!

In [ ]:
## Example chai prediction command

qlogin -c4 --mem=48g -p gpu --gres=gpu:a6000:1

/net/software/lab/chai/chai-lab/run_chai.sh --structure_output pdb \
--pdb_folder /home/pkwon2/tmp/paired_filtered/chai \
--ligand "['[Ca++]','[Ca++]','[Ca++]','[Ca++]','[Ca++]','[Ca++]','[Ca++]','[Ca++]','[Ca++]','[Ca++]','[Ca++]','[Ca++]']" \
--output_dir /home/pkwon2/tmp/paired_filtered/chai/aragonite_chaipreds/